<a href="https://colab.research.google.com/github/Major-project-BT3073/GAN-and-Diffusion-Augmentation-/blob/main/diffusion_and_gan_research_paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harnessing the Power of Diffusion Models for Plant Disease Image Augmentation
**Paper**: Muhammad et al., Frontiers in Plant Science, 2023

This notebook implements:
1. PlantVillage dataset setup
2. U-Net leaf segmentation (ResNet-50 backbone)
3. InstaGAN (GAN-based augmentation)
4. RePaint (Diffusion-based augmentation)
5. Evaluation: FID, KID, PSNR, SSIM, IS

## Cell 1 — Install Dependencies

In [ ]:
# ── Install all required packages ──────────────────────────────────────────
import subprocess, sys

pkgs = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118",
    "segmentation-models-pytorch",
    "albumentations",
    "clean-fid",
    "torch-fidelity",
    "diffusers==0.27.2",
    "accelerate",
    "opendatasets",
    "timm",
    "matplotlib seaborn pandas scikit-learn tqdm Pillow opencv-python-headless",
]
for p in pkgs:
    subprocess.run(f"{sys.executable} -m pip install -q {p}", shell=True)

print("✅ All packages installed")

✅ All packages installed


## Cell 2 — Imports & GPU Check

In [ ]:
import os, random, time, warnings, math, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid, save_image
import torchvision.models as models
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥  Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(SEED)

# ── Global paths ────────────────────────────────────────────────────────────
ROOT       = Path('/content')
DATA_DIR   = ROOT / 'plantvillage'
SEG_DIR    = ROOT / 'seg_masks'
GAN_DIR    = ROOT / 'gan_outputs'
DIFF_DIR   = ROOT / 'diffusion_outputs'
CKPT_DIR   = ROOT / 'checkpoints'
for d in [DATA_DIR, SEG_DIR, GAN_DIR, DIFF_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("✅ Imports OK — directories created")

🖥  Device : cpu
✅ Imports OK — directories created


## Cell 3 — Download PlantVillage Dataset (Kaggle)

In [ ]:
# ── Option A: Kaggle API (recommended) ─────────────────────────────────────
# Upload your kaggle.json first OR use the manual link below.

import opendatasets as od

# This will prompt for Kaggle username & API key once
dataset_url = "https://www.kaggle.com/datasets/mohitsingh1804/plantvillage"
od.download(dataset_url, data_dir=str(ROOT))

# ── Auto-detect the extracted folder ───────────────────────────────────────
candidates = list(ROOT.glob('**/PlantVillage')) + list(ROOT.glob('**/plantvillage'))
if not candidates:
    candidates = [p for p in ROOT.iterdir() if p.is_dir() and 'plant' in p.name.lower()]

PLANT_VILLAGE = candidates[0] if candidates else ROOT / 'plantvillage'
print(f"📂 PlantVillage root : {PLANT_VILLAGE}")

# List all disease classes found
all_classes = sorted([d.name for d in PLANT_VILLAGE.rglob('*') if d.is_dir() and
                       any(f.suffix.lower() in ['.jpg','.jpeg','.png'] for f in d.iterdir())])
print(f"   Found {len(all_classes)} classes")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:

## Cell 4 — Define Target Classes & Build Dataset Index

In [ ]:
# ── Target classes matching Table 1 of the paper ───────────────────────────
TARGET_CLASSES = {
    # Tomato diseases
    'Tomato___Late_blight':                  ('Tomato', 'Late Blight',              1000),
    'Tomato___Early_blight':                 ('Tomato', 'Early Blight',              800),
    'Tomato___Septoria_leaf_spot':           ('Tomato', 'Septoria Leaf Spot',        700),
    'Tomato___Target_Spot':                  ('Tomato', 'Target Spot',               600),
    'Tomato___Tomato_mosaic_virus':          ('Tomato', 'Mosaic Virus',              500),
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus':('Tomato', 'Yellow Leaf Curl Virus',   400),
    'Tomato___Spider_mites Two-spotted_spider_mite': ('Tomato', 'Spider Mites',     300),
    'Tomato___Leaf_Mold':                    ('Tomato', 'Leaf Mold',                200),
    'Tomato___Bacterial_spot':               ('Tomato', 'Bacterial Spot',           100),
    'Tomato___healthy':                      ('Tomato', 'Healthy',                 1200),
    # Grape diseases
    'Grape___Black_rot':                     ('Grape',  'Black Rot',                500),
    'Grape___Esca_(Black_Measles)':          ('Grape',  'Esca (Black Measles)',     400),
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': ('Grape', 'Leaf Blight',         300),
    'Grape___healthy':                       ('Grape',  'Healthy',                  600),
}

def find_class_dir(plant_village_root, class_key):
    """Fuzzy match class directory names."""
    for d in plant_village_root.rglob('*'):
        if d.is_dir() and class_key.lower().replace('___','_') in d.name.lower().replace(' ','_'):
            return d
    # Try partial match
    key_parts = class_key.lower().split('___')
    for d in plant_village_root.rglob('*'):
        if d.is_dir():
            dn = d.name.lower()
            if all(p.replace('_',' ') in dn.replace('_',' ') for p in key_parts):
                return d
    return None

image_index = []   # list of dicts: {path, plant, disease, label_idx}
label_map   = {}
label_idx   = 0

for cls_key, (plant, disease, _) in TARGET_CLASSES.items():
    cls_dir = find_class_dir(PLANT_VILLAGE, cls_key)
    if cls_dir is None:
        # fallback: search by plant + disease keyword
        kw = disease.lower().replace(' ','_')
        for d in PLANT_VILLAGE.rglob('*'):
            if d.is_dir() and plant.lower() in d.name.lower() and kw[:5] in d.name.lower():
                cls_dir = d; break
    if cls_dir is None:
        print(f"  ⚠ Not found: {cls_key}")
        continue

    imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in ['.jpg','.jpeg','.png']]
    label_map[cls_key] = label_idx
    for img_path in imgs:
        image_index.append({
            'path': str(img_path), 'plant': plant,
            'disease': disease, 'cls_key': cls_key,
            'label': label_idx
        })
    print(f"  ✔ {plant:6s} | {disease:30s} | {len(imgs):5d} images | dir={cls_dir.name}")
    label_idx += 1

df = pd.DataFrame(image_index)
print(f"\n📊 Total images indexed: {len(df)}")

## Cell 5 — Dataset Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
counts = df.groupby(['plant','disease']).size().reset_index(name='count')
colors = {'Tomato': '#e74c3c', 'Grape': '#8e44ad'}
bar_colors = [colors[p] for p in counts['plant']]
axes[0].barh(counts['disease'], counts['count'], color=bar_colors)
axes[0].set_xlabel('Number of Images')
axes[0].set_title('PlantVillage — Selected Classes (Table 1)')
patches = [mpatches.Patch(color=v, label=k) for k,v in colors.items()]
axes[0].legend(handles=patches)

# Pie chart
plant_counts = df.groupby('plant').size()
axes[1].pie(plant_counts, labels=plant_counts.index, autopct='%1.1f%%',
            colors=['#e74c3c','#8e44ad'])
axes[1].set_title('Tomato vs Grape Split')

plt.tight_layout()
plt.savefig('/content/dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total: Tomato={len(df[df.plant=='Tomato'])} | Grape={len(df[df.plant=='Grape'])}")

## Cell 6 — Leaf Segmentation: U-Net (ResNet-50 Backbone)

In [ ]:
# ── U-Net with ResNet-50 backbone via segmentation_models_pytorch ──────────
SEG_IMG_SIZE = 256

seg_model = smp.Unet(
    encoder_name    = 'resnet50',
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = 1,
    activation      = 'sigmoid'
).to(DEVICE)

print(f"✅ U-Net (ResNet-50) loaded")
total_params = sum(p.numel() for p in seg_model.parameters())
print(f"   Parameters: {total_params/1e6:.1f}M")

## Cell 7 — Segmentation Dataset & Transforms

In [ ]:
class LeafSegDataset(Dataset):
    """
    Returns (image_tensor, mask_tensor).
    If no real mask exists, generates an Otsu threshold mask on-the-fly
    (used as pseudo-label for unsupervised pre-training).
    """
    def __init__(self, image_paths, mask_paths=None, img_size=256, augment=False):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths  # None → auto Otsu
        self.img_size    = img_size
        self.augment     = augment

        self.img_tfm = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5 if augment else 0),
            A.RandomBrightnessContrast(p=0.3 if augment else 0),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        self.msk_tfm = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5 if augment else 0),
        ])

    def _otsu_mask(self, img_bgr):
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5,5), 0)
        _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        # keep largest connected component
        n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        if n > 1:
            largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
            mask = (labels == largest).astype(np.uint8) * 255
        return mask

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img_bgr = cv2.imread(str(self.image_paths[idx]))
        img_bgr = cv2.resize(img_bgr, (self.img_size, self.img_size))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.mask_paths is not None:
            msk = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, (self.img_size, self.img_size))
        else:
            msk = self._otsu_mask(img_bgr)

        augmented = self.img_tfm(image=img_rgb)
        img_t = augmented['image']                     # [3, H, W] float32
        msk_t = torch.from_numpy(msk).float() / 255.0 # [H, W]
        msk_t = msk_t.unsqueeze(0)                     # [1, H, W]
        return img_t, msk_t


# ── Build dataset from indexed images (no manual masks needed) ─────────────
all_paths = [Path(r['path']) for _, r in df.iterrows()]
random.shuffle(all_paths)
split = int(0.85 * len(all_paths))
train_paths, val_paths = all_paths[:split], all_paths[split:]

seg_train = LeafSegDataset(train_paths, augment=True)
seg_val   = LeafSegDataset(val_paths,   augment=False)

seg_train_loader = DataLoader(seg_train, batch_size=16, shuffle=True,
                               num_workers=2, pin_memory=True)
seg_val_loader   = DataLoader(seg_val,   batch_size=16, shuffle=False,
                               num_workers=2, pin_memory=True)

print(f"✅ Seg dataset — Train: {len(seg_train)} | Val: {len(seg_val)}")

## Cell 8 — Train U-Net Segmentation Model

In [ ]:
# ── CELL A: Find your data location ──────────────────────────
import os

# Search everywhere
print("Searching for PlantVillage images...")
for root, dirs, files in os.walk('/content'):
    imgs = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if len(imgs) > 50:
        print(f"  Found {len(imgs):5d} images in: {root}")

In [ ]:
# ===================== IMPORTS =====================
import os, torch, torch.nn.functional as F
import numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import torch.optim as optim
from joblib import Parallel, delayed
import segmentation_models_pytorch as smp

# ===================== DEVICE =====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("🔥 Using:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# ===================== DATA PATH =====================
DATA = Path('/content/plantvillage/PlantVillage')

# ===================== BALANCED CONFIG =====================
SEG_SIZE = 160
SEG_BATCH = 8
SEG_LR = 1e-4
MAX_PER_CLASS = 100

SEG_EPOCHS_STAGE1 = 2
SEG_EPOCHS_STAGE2 = 2

# ===================== CLASS MAP =====================
WANT = {
    'Grape___Black_rot':'grape_black_rot',
    'Grape___Esca_(Black_Measles)':'grape_black_measles',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)':'grape_leaf_blight',
    'Grape___healthy':'grape_healthy',
    'Tomato___Late_blight':'tomato_late_blight',
    'Tomato___Early_blight':'tomato_early_blight',
    'Tomato___Septoria_leaf_spot':'tomato_septoria',
    'Tomato___Target_Spot':'tomato_target_spot',
    'Tomato___Tomato_mosaic_virus':'tomato_mosaic_virus',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus':'tomato_yellow_curl',
    'Tomato___Spider_mites Two-spotted_spider_mite':'tomato_spider_mites',
    'Tomato___Leaf_Mold':'tomato_leaf_mold',
    'Tomato___Bacterial_spot':'tomato_bacterial_spot',
    'Tomato___healthy':'tomato_healthy',
}

# ===================== LOAD DATA =====================
all_paths = []

for split in ['train','val']:
    sp = DATA/split
    if not sp.exists(): continue

    for k in WANT:
        cp = sp/k
        if cp.exists():
            imgs = sorted([p for p in cp.iterdir()
                           if p.suffix.lower() in ['.jpg','.png','.jpeg']])
            all_paths += imgs[:MAX_PER_CLASS]

print("Total images:", len(all_paths))
if len(all_paths) == 0:
    raise ValueError("❌ Dataset not found")

# ===================== SPLIT =====================
split_idx = int(len(all_paths)*0.9)
train_paths = all_paths[:split_idx]
val_paths   = all_paths[split_idx:]

# ===================== MASK =====================
def make_mask(path):
    try:
        img = cv2.resize(np.array(Image.open(path).convert('RGB')), (SEG_SIZE, SEG_SIZE))
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, m = cv2.threshold(cv2.GaussianBlur(gray,(5,5),0),0,255,
                             cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
        return (m>127).astype(np.float32)
    except:
        return np.zeros((SEG_SIZE,SEG_SIZE),np.float32)

print("⚡ Generating masks...")
all_masks = Parallel(n_jobs=-1)(delayed(make_mask)(str(p)) for p in tqdm(all_paths))
MASK_CACHE = {str(p):m for p,m in zip(all_paths, all_masks)}

# ===================== DATASET =====================
class DS(Dataset):
    def __init__(self, paths, train=True):
        self.paths = paths

        if train:
            self.tf = transforms.Compose([
                transforms.Resize((SEG_SIZE,SEG_SIZE)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize([0.5]*3,[0.5]*3)
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((SEG_SIZE,SEG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize([0.5]*3,[0.5]*3)
            ])

    def __len__(self): return len(self.paths)

    def __getitem__(self,i):
        p = self.paths[i]
        img = self.tf(Image.open(p).convert('RGB'))
        mask = torch.tensor(MASK_CACHE[str(p)]).unsqueeze(0)
        return img, mask

train_loader = DataLoader(DS(train_paths,True), batch_size=SEG_BATCH, shuffle=True)
val_loader   = DataLoader(DS(val_paths,False), batch_size=SEG_BATCH)

# ===================== LOSS =====================
def dice_loss(x,y):
    x = torch.sigmoid(x).view(-1)
    y = y.view(-1)
    return 1-(2*(x*y).sum()+1)/(x.sum()+y.sum()+1)

def loss_fn(x,y):
    return 0.7*dice_loss(x,y)+0.3*F.binary_cross_entropy_with_logits(x,y)

def miou(x,y):
    p=(torch.sigmoid(x)>0.35).float()
    i=(p*y).sum()
    u=p.sum()+y.sum()-i
    return (i/(u+1e-8)).item()

# ===================== MODEL =====================
model = smp.Unet('mobilenet_v2',encoder_weights='imagenet',in_channels=3,classes=1).to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=SEG_LR)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

# ===================== PSEUDO =====================
def gen_pseudo(model,paths):
    model.eval()
    new = {}
    tf = transforms.Compose([
        transforms.Resize((SEG_SIZE,SEG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3,[0.5]*3)
    ])
    with torch.no_grad():
        for p in paths:
            x = tf(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
            pr = torch.sigmoid(model(x))[0,0].cpu().numpy()
            new[str(p)] = (pr>0.35).astype(np.float32)
    return new

# ===================== TRAIN =====================
best=0

print("\n🚀 Stage 1")
for e in range(SEG_EPOCHS_STAGE1):
    model.train()
    for img,mask in tqdm(train_loader):
        img,mask=img.to(DEVICE),mask.to(DEVICE)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
            loss=loss_fn(model(img),mask)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

print("\n⚡ Pseudo Update")
MASK_CACHE.update(gen_pseudo(model,train_paths))

print("\n🚀 Stage 2")
for e in range(SEG_EPOCHS_STAGE2):
    model.train()
    for img,mask in tqdm(train_loader):
        img,mask=img.to(DEVICE),mask.to(DEVICE)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
            loss=loss_fn(model(img),mask)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

    model.eval()
    score=0
    with torch.no_grad():
        for img,mask in val_loader:
            img,mask=img.to(DEVICE),mask.to(DEVICE)
            score+=miou(model(img),mask)

    score=(score/len(val_loader))*100
    print(f"Epoch {e+1} mIoU {score:.2f}%")

    if score>best:
        best=score

print("\n✅ FINAL mIoU:",best)

## Cell 9 — Generate & Save Segmentation Masks

In [ ]:
seg_model.load_state_dict(torch.load(CKPT_DIR / 'unet_best.pth'))
seg_model.eval()

MASK_SIZE = 256
img_norm  = T.Compose([
    T.Resize((MASK_SIZE, MASK_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def predict_mask(img_path):
    """Returns binary numpy mask [H, W] uint8 (0/255)."""
    img = Image.open(img_path).convert('RGB')
    inp = img_norm(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = seg_model(inp).squeeze().cpu().numpy()
    mask = (pred > 0.5).astype(np.uint8) * 255
    return mask

def dilate_mask_for_repaint(mask, kernel_size=15):
    """Dilate & invert for RePaint (paper §5.2.2)."""
    kernel  = np.ones((kernel_size, kernel_size), np.uint8)
    dilated = cv2.dilate(mask, kernel, iterations=2)
    return dilated

# Generate masks for a subset
MASK_SUBSET = 200  # generate for first N images per class
print("Generating segmentation masks...")
mask_records = []

for cls_key, group in tqdm(df.groupby('cls_key'), desc='Classes'):
    cls_mask_dir = SEG_DIR / cls_key
    cls_mask_dir.mkdir(exist_ok=True)
    for _, row in group.head(MASK_SUBSET).iterrows():
        img_path  = Path(row['path'])
        mask_path = cls_mask_dir / (img_path.stem + '_mask.png')
        if not mask_path.exists():
            m = predict_mask(img_path)
            cv2.imwrite(str(mask_path), m)
        mask_records.append({'img': str(img_path), 'mask': str(mask_path),
                              'plant': row['plant'], 'disease': row['disease']})

mask_df = pd.DataFrame(mask_records)
print(f"✅ Masks generated: {len(mask_df)}")

## Cell 10 — Visualize Segmentation Results

In [ ]:
# Show Otsu vs U-Net masks (Figure 6 of paper)
sample_imgs = mask_df.sample(4, random_state=42)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Segmentation: Original | Otsu Threshold | U-Net Mask', fontsize=13)

for col, (_, row) in enumerate(sample_imgs.iterrows()):
    img_bgr = cv2.imread(row['img'])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_rgb = cv2.resize(img_rgb, (256, 256))

    # Otsu mask
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blur    = cv2.GaussianBlur(gray, (5,5), 0)
    _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    otsu    = cv2.resize(otsu, (256, 256))

    # U-Net mask
    unet_mask = cv2.imread(row['mask'], cv2.IMREAD_GRAYSCALE)

    axes[0][col].imshow(img_rgb);      axes[0][col].axis('off')
    axes[1][col].imshow(otsu, cmap='gray'); axes[1][col].axis('off')
    axes[2][col].imshow(unet_mask, cmap='gray'); axes[2][col].axis('off')

    axes[0][col].set_title(f"{row['disease'][:20]}", fontsize=8)

axes[0][0].set_ylabel('Original', fontsize=10)
axes[1][0].set_ylabel('Otsu Thresh', fontsize=10)
axes[2][0].set_ylabel('U-Net Mask', fontsize=10)

plt.tight_layout()
plt.savefig('/content/segmentation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Report mIoU
print(f"\nSegmentation Results (Table 2 in paper):")
print(f"  Otsu Threshold        : 75.36% mIoU")
print(f"  U-Net (ResNet-50 bkb) : {best_miou:.2f}% mIoU")

## Cell 11 — InstaGAN: Architecture Definition

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# InstaGAN — Instance-aware GAN (Mo et al., 2019)                            #
# Based on CycleGAN + conditional GAN + instance segmentation masks           #
# Paper: §4.1 + Equations 1-9                                                 #
# ─────────────────────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3), nn.InstanceNorm2d(dim), nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3), nn.InstanceNorm2d(dim)
        )
    def forward(self, x): return x + self.block(x)


class InstaGenerator(nn.Module):
    """
    Generator G_{XY}: (image + mask) → (translated image + translated mask).
    Implements Equations 1-2 of the paper.
    Input channels = 4 (RGB image + 1 mask channel).
    """
    def __init__(self, in_ch=4, out_ch=4, ngf=64, n_res=9):
        super().__init__()
        # Encoder
        enc = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True),
            nn.Conv2d(ngf,   ngf*2, 3, 2, 1), nn.InstanceNorm2d(ngf*2), nn.ReLU(True),
            nn.Conv2d(ngf*2, ngf*4, 3, 2, 1), nn.InstanceNorm2d(ngf*4), nn.ReLU(True),
        ]
        # Residual blocks
        res = [ResidualBlock(ngf*4) for _ in range(n_res)]
        # Decoder
        dec = [
            nn.ConvTranspose2d(ngf*4, ngf*2, 3, 2, 1, 1), nn.InstanceNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   3, 2, 1, 1), nn.InstanceNorm2d(ngf),   nn.ReLU(True),
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, out_ch, 7), nn.Tanh()
        ]
        self.net = nn.Sequential(*enc, *res, *dec)

    def forward(self, img, mask):
        x = torch.cat([img, mask], dim=1)   # (B, 4, H, W)
        out = self.net(x)                   # (B, 4, H, W)
        return out[:, :3], out[:, 3:4]      # (fake_img, fake_mask)


class InstaDiscriminator(nn.Module):
    """
    PatchGAN discriminator. Permutation-invariant to instances (Eq. 3).
    Input: (image, mask) → 4 channels.
    """
    def __init__(self, in_ch=4, ndf=64):
        super().__init__()
        def block(ic, oc, stride=2, norm=True):
            layers = [nn.Conv2d(ic, oc, 4, stride, 1, bias=not norm)]
            if norm: layers.append(nn.InstanceNorm2d(oc))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers
        self.net = nn.Sequential(
            *block(in_ch,   ndf,   norm=False),
            *block(ndf,     ndf*2),
            *block(ndf*2,   ndf*4),
            *block(ndf*4,   ndf*8, stride=1),
            nn.Conv2d(ndf*8, 1, 4, 1, 1)
        )
    def forward(self, img, mask):
        return self.net(torch.cat([img, mask], dim=1))


print("✅ InstaGAN architecture defined")
# Quick shape test
_g = InstaGenerator().to(DEVICE)
_i = torch.randn(1, 3, 200, 200).to(DEVICE)
_m = torch.randn(1, 1, 200, 200).to(DEVICE)
_fi, _fm = _g(_i, _m)
print(f"   Generator output: img={tuple(_fi.shape)}, mask={tuple(_fm.shape)}")
del _g, _i, _m, _fi, _fm

## Cell 12 — InstaGAN: Loss Functions (Equations 4-9)

In [ ]:
class InstaGANLoss(nn.Module):
    """Combined LSGAN + Cycle + Identity + Context loss (Eq. 4-8)."""

    def __init__(self, lam_cyc=10.0, lam_idt=5.0, lam_ctx=10.0):
        super().__init__()
        self.lam_cyc = lam_cyc
        self.lam_idt = lam_idt
        self.lam_ctx = lam_ctx
        self.mse     = nn.MSELoss()
        self.l1      = nn.L1Loss()

    def lsgan_loss(self, disc_real, disc_fake):
        """Eq. 4 — Least Squares GAN loss."""
        real_loss = self.mse(disc_real, torch.ones_like(disc_real))
        fake_loss = self.mse(disc_fake, torch.zeros_like(disc_fake))
        return (real_loss + fake_loss) * 0.5

    def gen_lsgan_loss(self, disc_fake):
        return self.mse(disc_fake, torch.ones_like(disc_fake))

    def cycle_loss(self, real_img, real_mask, rec_img, rec_mask):
        """Eq. 5 — Cycle consistency loss."""
        return self.l1(rec_img, real_img) + self.l1(rec_mask, real_mask)

    def identity_loss(self, real_img, real_mask, idt_img, idt_mask):
        """Eq. 6 — Identity mapping loss."""
        return self.l1(idt_img, real_img) + self.l1(idt_mask, real_mask)

    def context_loss(self, real_img, fake_img, mask_a, mask_b):
        """Eq. 7 — Context preserving loss (background preservation)."""
        # weight = 1 - mask (background regions)
        w = 1.0 - torch.clamp(mask_a + mask_b, 0, 1)
        return self.l1(w * real_img, w * fake_img)

    def total_generator_loss(self, G_XY, G_YX,
                              D_X, D_Y,
                              real_X, real_A,
                              real_Y, real_B):
        """Eq. 8 — Full InstaGAN generator loss."""
        # Forward translation X → Y
        fake_Y, fake_B = G_XY(real_X, real_A)
        # Forward translation Y → X
        fake_X, fake_A = G_YX(real_Y, real_B)
        # Cycle reconstruction
        rec_X, rec_A   = G_YX(fake_Y, fake_B)
        rec_Y, rec_B   = G_XY(fake_X, fake_A)
        # Identity (optional but helps stability)
        idt_Y, idt_B   = G_XY(real_Y, real_B)
        idt_X, idt_A   = G_YX(real_X, real_A)

        # Discriminator predictions on fakes
        d_fake_Y = D_Y(fake_Y.detach(), fake_B.detach())
        d_fake_X = D_X(fake_X.detach(), fake_A.detach())

        loss_G = (
            self.gen_lsgan_loss(D_Y(fake_Y, fake_B)) +
            self.gen_lsgan_loss(D_X(fake_X, fake_A)) +
            self.lam_cyc * (self.cycle_loss(real_X, real_A, rec_X, rec_A) +
                            self.cycle_loss(real_Y, real_B, rec_Y, rec_B)) +
            self.lam_idt * (self.identity_loss(real_Y, real_B, idt_Y, idt_B) +
                            self.identity_loss(real_X, real_A, idt_X, idt_A)) +
            self.lam_ctx * (self.context_loss(real_X, fake_Y, real_A, fake_B) +
                            self.context_loss(real_Y, fake_X, real_B, fake_A))
        )
        return loss_G, fake_Y, fake_B, fake_X, fake_A

print("✅ InstaGAN loss functions defined (Eq. 4-9)")

## Cell 13 — InstaGAN: Dataset & Training Loop

In [ ]:
class InstaGANDataset(Dataset):
    """Pairs (healthy image + mask) with (diseased image + mask)."""
    def __init__(self, healthy_df, disease_df, mask_dir, img_size=200):
        self.healthy  = healthy_df.reset_index(drop=True)
        self.disease  = disease_df.reset_index(drop=True)
        self.mask_dir = Path(mask_dir)
        self.tfm = T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])
        self.mask_tfm = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor()
        ])

    def _load_pair(self, row):
        img  = Image.open(row['path']).convert('RGB')
        # Look for pre-generated mask
        cls  = row['cls_key']
        stem = Path(row['path']).stem
        mp   = self.mask_dir / cls / f"{stem}_mask.png"
        if mp.exists():
            mask = Image.open(mp).convert('L')
        else:
            # Otsu fallback
            bg   = cv2.imread(row['path'])
            gray = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)
            blur = cv2.GaussianBlur(gray, (5,5), 0)
            _, m = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            mask = Image.fromarray(m)
        return self.tfm(img), self.mask_tfm(mask)

    def __len__(self): return max(len(self.healthy), len(self.disease))

    def __getitem__(self, idx):
        h_row = self.healthy.iloc[idx % len(self.healthy)]
        d_row = self.disease.iloc[idx % len(self.disease)]
        img_X, mask_A = self._load_pair(h_row)
        img_Y, mask_B = self._load_pair(d_row)
        return img_X, mask_A, img_Y, mask_B


# ── Build datasets for one disease class (Early Blight example) ────────────
healthy_tomato = df[df['cls_key'] == 'Tomato___healthy'].copy()
early_blight   = df[df['cls_key'] == 'Tomato___Early_blight'].copy()

if len(healthy_tomato) == 0 or len(early_blight) == 0:
    # Use whatever we have
    healthy_tomato = df[df['disease'].str.lower().str.contains('healthy') &
                        df['plant'].str.lower().str.contains('tomato')].copy()
    early_blight   = df[df['plant'].str.lower().str.contains('tomato') &
                        ~df['disease'].str.lower().str.contains('healthy')].head(400).copy()

insta_ds     = InstaGANDataset(healthy_tomato, early_blight, SEG_DIR)
insta_loader = DataLoader(insta_ds, batch_size=1, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
print(f"✅ InstaGAN dataset — {len(insta_ds)} pairs")

# ── Instantiate networks ───────────────────────────────────────────────────
G_XY = InstaGenerator().to(DEVICE)   # healthy → diseased
G_YX = InstaGenerator().to(DEVICE)   # diseased → healthy
D_X  = InstaDiscriminator().to(DEVICE)
D_Y  = InstaDiscriminator().to(DEVICE)

insta_loss = InstaGANLoss()
lr         = 2e-4
opt_G      = optim.Adam(list(G_XY.parameters()) + list(G_YX.parameters()),
                         lr=lr, betas=(0.5, 0.999))
opt_D      = optim.Adam(list(D_X.parameters())  + list(D_Y.parameters()),
                         lr=lr, betas=(0.5, 0.999))

GAN_EPOCHS = 50   # paper uses 100+; 50 for demo
gan_history = {'G_loss': [], 'D_loss': []}

print(f"Starting InstaGAN training for {GAN_EPOCHS} epochs...")
for epoch in range(1, GAN_EPOCHS + 1):
    G_losses, D_losses = [], []

    for real_X, real_A, real_Y, real_B in tqdm(
            insta_loader, desc=f'GAN Ep {epoch}/{GAN_EPOCHS}', leave=False):
        real_X = real_X.to(DEVICE); real_A = real_A.to(DEVICE)
        real_Y = real_Y.to(DEVICE); real_B = real_B.to(DEVICE)

        # ── Generator step ─────────────────────────────────────────────────
        opt_G.zero_grad()
        loss_G, fake_Y, fake_B, fake_X, fake_A = insta_loss.total_generator_loss(
            G_XY, G_YX, D_X, D_Y, real_X, real_A, real_Y, real_B)
        loss_G.backward()
        opt_G.step()

        # ── Discriminator step ─────────────────────────────────────────────
        opt_D.zero_grad()
        fake_Y2, fake_B2 = G_XY(real_X, real_A)
        fake_X2, fake_A2 = G_YX(real_Y, real_B)
        loss_DY = insta_loss.lsgan_loss(D_Y(real_Y, real_B),
                                         D_Y(fake_Y2.detach(), fake_B2.detach()))
        loss_DX = insta_loss.lsgan_loss(D_X(real_X, real_A),
                                         D_X(fake_X2.detach(), fake_A2.detach()))
        loss_D  = loss_DX + loss_DY
        loss_D.backward()
        opt_D.step()

        G_losses.append(loss_G.item())
        D_losses.append(loss_D.item())

    g = np.mean(G_losses); d = np.mean(D_losses)
    gan_history['G_loss'].append(g)
    gan_history['D_loss'].append(d)
    if epoch % 10 == 0:
        print(f"  Ep {epoch:3d} | G_loss={g:.4f} | D_loss={d:.4f}")
        torch.save({'G_XY': G_XY.state_dict(), 'G_YX': G_YX.state_dict()},
                   CKPT_DIR / f'instagan_ep{epoch}.pth')

print("✅ InstaGAN training complete")

## Cell 14 — RePaint: DDPM Implementation (Equations 10-16)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RePaint — DDPM-based inpainting (Lugmayr et al., 2022)                     #
# Paper §4.2 — Equations 10-16                                               #
# ─────────────────────────────────────────────────────────────────────────────

class SinusoidalPE(nn.Module):
    """Sinusoidal positional embedding for diffusion timestep."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half   = self.dim // 2
        freqs  = torch.exp(-math.log(10000) *
                            torch.arange(half, device=t.device) / half)
        args   = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, ch, t_emb_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, 1, 1)
        self.t_proj = nn.Linear(t_emb_dim, ch)
        self.norm2 = nn.GroupNorm(8, ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, 1, 1)
        self.act   = nn.SiLU()

    def forward(self, x, t_emb):
        h  = self.act(self.norm1(x))
        h  = self.conv1(h)
        h  = h + self.t_proj(self.act(t_emb))[:, :, None, None]
        h  = self.act(self.norm2(h))
        h  = self.conv2(h)
        return x + h


class UNetDDPM(nn.Module):
    """
    Lightweight U-Net for DDPM epsilon prediction.
    Input: noisy image x_t (B,3,H,W) + timestep t → epsilon (B,3,H,W)
    """
    def __init__(self, img_ch=3, base_ch=64, t_emb_dim=128):
        super().__init__()
        self.t_emb = nn.Sequential(
            SinusoidalPE(t_emb_dim),
            nn.Linear(t_emb_dim, t_emb_dim*2), nn.SiLU(),
            nn.Linear(t_emb_dim*2, t_emb_dim)
        )
        ch = base_ch
        # Encoder
        self.enc1 = nn.Conv2d(img_ch, ch, 3, 1, 1)
        self.rb1  = ResBlock(ch,    t_emb_dim)
        self.down1 = nn.Conv2d(ch, ch*2, 4, 2, 1)
        self.rb2  = ResBlock(ch*2, t_emb_dim)
        self.down2 = nn.Conv2d(ch*2, ch*4, 4, 2, 1)
        # Bottleneck
        self.rb3  = ResBlock(ch*4, t_emb_dim)
        self.rb4  = ResBlock(ch*4, t_emb_dim)
        # Decoder
        self.up1  = nn.ConvTranspose2d(ch*4, ch*2, 4, 2, 1)
        self.rb5  = ResBlock(ch*4, t_emb_dim)   # skip from enc
        self.up2  = nn.ConvTranspose2d(ch*4, ch, 4, 2, 1)
        self.rb6  = ResBlock(ch*2, t_emb_dim)   # skip
        self.out  = nn.Sequential(
            nn.GroupNorm(8, ch*2), nn.SiLU(),
            nn.Conv2d(ch*2, img_ch, 3, 1, 1)
        )

    def forward(self, x, t):
        te   = self.t_emb(t)
        e1   = self.rb1(self.enc1(x), te)              # (B, ch, H, W)
        e2   = self.rb2(self.down1(e1), te)            # (B, ch*2, H/2, W/2)
        e3   = self.rb3(self.down2(e2), te)            # (B, ch*4, H/4, W/4)
        e3   = self.rb4(e3, te)
        d1   = self.rb5(torch.cat([self.up1(e3), e2], 1), te)
        d2   = self.rb6(torch.cat([self.up2(d1), e1], 1), te)
        return self.out(d2)


class DDPM:
    """
    Denoising Diffusion Probabilistic Model.
    Implements forward (Eq.10, 13) and reverse (Eq.11, 12) processes.
    RePaint inpainting (Eq. 14-16).
    """
    def __init__(self, model, T=1000, beta_start=1e-4, beta_end=2e-2, device='cpu'):
        self.model  = model
        self.T      = T
        self.device = device

        # Variance schedule (linear)
        betas           = torch.linspace(beta_start, beta_end, T).to(device)
        alphas          = 1.0 - betas
        alphas_bar      = torch.cumprod(alphas, 0)
        alphas_bar_prev = F.pad(alphas_bar[:-1], (1,0), value=1.0)

        self.betas           = betas
        self.alphas          = alphas
        self.alphas_bar      = alphas_bar           # ᾱ_t  (Eq. 13)
        self.alphas_bar_prev = alphas_bar_prev
        self.sqrt_ab         = alphas_bar.sqrt()
        self.sqrt_one_minus_ab = (1 - alphas_bar).sqrt()
        # Posterior variance for reverse step
        self.post_var = betas * (1 - alphas_bar_prev) / (1 - alphas_bar)

    def q_sample(self, x0, t, noise=None):
        """Forward diffusion: Eq. 10 / 13 — add noise to x0."""
        if noise is None: noise = torch.randn_like(x0)
        sa  = self.sqrt_ab[t][:, None, None, None]
        soa = self.sqrt_one_minus_ab[t][:, None, None, None]
        return sa * x0 + soa * noise, noise

    def p_sample(self, xt, t_idx):
        """Single reverse step: Eq. 11 — denoise by one step."""
        t_tensor = torch.full((xt.shape[0],), t_idx, device=self.device, dtype=torch.long)
        with torch.no_grad():
            eps_pred = self.model(xt, t_tensor)
        beta     = self.betas[t_idx]
        alpha    = self.alphas[t_idx]
        ab       = self.alphas_bar[t_idx]
        mean     = (1/alpha.sqrt()) * (xt - beta / (1-ab).sqrt() * eps_pred)
        if t_idx > 0:
            noise = torch.randn_like(xt)
            var   = self.post_var[t_idx]
            x_prev = mean + var.sqrt() * noise
        else:
            x_prev = mean
        return x_prev

    def repaint_sample(self, x0, mask, n_resample=10, jump_len=10):
        """
        RePaint inpainting (Eq. 14-16).
        mask = 1 for known (unmasked) regions, 0 for unknown (to inpaint).
        Implements the resampling trick for semantic consistency.
        """
        self.model.eval()
        B, C, H, W = x0.shape
        x0   = x0.to(self.device)
        mask = mask.to(self.device)       # (B,1,H,W) — 1=keep, 0=inpaint

        # Start from pure noise
        xt = torch.randn_like(x0)

        t_seq = list(range(self.T - 1, -1, -1))
        i = 0
        pbar = tqdm(total=self.T, desc='RePaint', leave=False)
        while i < len(t_seq):
            t_idx = t_seq[i]
            t_tensor = torch.full((B,), t_idx, device=self.device, dtype=torch.long)

            # Eq.14: known region from forward process at t
            x_known, _ = self.q_sample(x0, t_tensor)
            # Eq.15: unknown region from reverse step
            x_unknown  = self.p_sample(xt, t_idx)
            # Eq.16: combine
            xt = mask * x_known + (1 - mask) * x_unknown

            # Resampling: jump back for semantic consistency
            if i < len(t_seq) - jump_len and t_idx > 0:
                resample_t = min(t_idx + jump_len, self.T - 1)
                noise = torch.randn_like(xt)
                ab    = self.alphas_bar[resample_t]
                xt    = ab.sqrt() * xt + (1-ab).sqrt() * noise
                i    = max(0, i - jump_len)  # jump back
            i += 1
            pbar.update(1)
        pbar.close()
        return xt.clamp(-1, 1)

    def train_step(self, x0, optimizer):
        """Single training step: Eq. 12 — predict noise."""
        B = x0.shape[0]
        t = torch.randint(0, self.T, (B,), device=self.device)
        xt, noise = self.q_sample(x0, t)
        eps_pred  = self.model(xt, t)
        loss      = F.mse_loss(eps_pred, noise)  # L_simple (Eq.12)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        optimizer.step()
        return loss.item()


# ── Instantiate ─────────────────────────────────────────────────────────────
ddpm_net  = UNetDDPM(img_ch=3, base_ch=64, t_emb_dim=128).to(DEVICE)
ddpm      = DDPM(ddpm_net, T=1000, device=DEVICE)
total_p   = sum(p.numel() for p in ddpm_net.parameters())
print(f"✅ DDPM U-Net defined | Parameters: {total_p/1e6:.1f}M")

# Shape test
_x = torch.randn(1, 3, 64, 64).to(DEVICE)
_t = torch.randint(0, 1000, (1,)).to(DEVICE)
_o = ddpm_net(_x, _t)
print(f"   DDPM output shape: {tuple(_o.shape)}")  # should be (1,3,64,64)
del _x, _t, _o

## Cell 15 — Train DDPM Model

In [ ]:
class LeafImageDataset(Dataset):
    """Simple dataset returning (normalized) leaf images."""
    def __init__(self, paths, img_size=64):
        self.paths = paths
        self.tfm   = T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        return self.tfm(Image.open(str(self.paths[idx])).convert('RGB'))


DDPM_IMG_SIZE = 64   # 256 requires much more GPU memory; use 64 for speed
DDPM_EPOCHS   = 20
DDPM_BS       = 16

# Train on ALL disease images from both tomato and grape
disease_paths = [Path(r['path']) for _, r in
                 df[~df['disease'].str.lower().str.contains('healthy')].iterrows()]
random.shuffle(disease_paths)

ddpm_ds     = LeafImageDataset(disease_paths, img_size=DDPM_IMG_SIZE)
ddpm_loader = DataLoader(ddpm_ds, batch_size=DDPM_BS, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)

ddpm_opt     = optim.Adam(ddpm_net.parameters(), lr=2e-4)
ddpm_history = []

print(f"Training DDPM on {len(disease_paths)} disease images ...")
for epoch in range(1, DDPM_EPOCHS + 1):
    ddpm_net.train()
    losses = []
    for batch in tqdm(ddpm_loader, desc=f'DDPM Ep {epoch}/{DDPM_EPOCHS}', leave=False):
        x0 = batch.to(DEVICE)
        l  = ddpm.train_step(x0, ddpm_opt)
        losses.append(l)
    mean_l = np.mean(losses)
    ddpm_history.append(mean_l)
    if epoch % 5 == 0:
        print(f"  Ep {epoch:2d} | MSE Loss: {mean_l:.5f}")
        torch.save(ddpm_net.state_dict(), CKPT_DIR / f'ddpm_ep{epoch}.pth')

torch.save(ddpm_net.state_dict(), CKPT_DIR / 'ddpm_final.pth')
print("✅ DDPM training complete")

## Cell 16 — Generate Augmented Images (InstaGAN & RePaint)

In [ ]:
def generate_instagan_samples(G_XY, healthy_df, mask_dir, n=50, out_dir=GAN_DIR):
    """Generate N synthetic disease images using InstaGAN."""
    G_XY.eval()
    out_dir = Path(out_dir) / 'instagan'
    out_dir.mkdir(parents=True, exist_ok=True)

    tfm = T.Compose([
        T.Resize((200, 200)), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)
    ])
    inv_norm = T.Normalize([-1,-1,-1], [2,2,2])

    paths = []
    rows  = healthy_df.sample(min(n, len(healthy_df)), random_state=0).reset_index()
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc='InstaGAN gen'):
        img  = Image.open(row['path']).convert('RGB')
        img_t = tfm(img).unsqueeze(0).to(DEVICE)

        stem = Path(row['path']).stem
        cls  = row.get('cls_key', 'Tomato___healthy')
        mp   = mask_dir / cls / f"{stem}_mask.png"
        if mp.exists():
            mask = Image.open(mp).convert('L')
            mask_t = T.Compose([T.Resize((200,200)), T.ToTensor()])(mask).unsqueeze(0).to(DEVICE)
        else:
            mask_t = torch.ones(1,1,200,200).to(DEVICE)

        with torch.no_grad():
            fake_img, _ = G_XY(img_t, mask_t)

        out_path = out_dir / f'instagan_{i:04d}.png'
        save_image(fake_img * 0.5 + 0.5, str(out_path))
        paths.append(out_path)
    print(f"  ✅ InstaGAN: {len(paths)} images → {out_dir}")
    return paths


def generate_repaint_samples(ddpm, healthy_df, mask_dir, n=30, out_dir=DIFF_DIR):
    """Generate N synthetic disease images using RePaint."""
    ddpm.model.eval()
    out_dir = Path(out_dir) / 'repaint'
    out_dir.mkdir(parents=True, exist_ok=True)

    tfm     = T.Compose([T.Resize((DDPM_IMG_SIZE, DDPM_IMG_SIZE)),
                          T.ToTensor(), T.Normalize([0.5]*3,[0.5]*3)])
    msk_tfm = T.Compose([T.Resize((DDPM_IMG_SIZE, DDPM_IMG_SIZE)), T.ToTensor()])

    def make_split_mask(size, split='half'):
        """Simple geometric split masks (paper §5.2.2 Figure 7)."""
        m = torch.zeros(1, 1, size, size)
        if   split == 'half':   m[:,:,:size//2,:] = 1
        elif split == 'quart':  m[:,:,:size//2,:size//2] = 1
        elif split == 'center': m[:,:,size//4:3*size//4, size//4:3*size//4] = 1
        return m   # 1 = known region

    splits = ['half', 'quart', 'center']
    paths  = []
    rows   = healthy_df.sample(min(n, len(healthy_df)), random_state=1).reset_index()

    for i, row in tqdm(rows.iterrows(), total=len(rows), desc='RePaint gen'):
        img    = Image.open(row['path']).convert('RGB')
        img_t  = tfm(img).unsqueeze(0).to(DEVICE)
        mask   = make_split_mask(DDPM_IMG_SIZE, random.choice(splits))

        result = ddpm.repaint_sample(img_t, mask, n_resample=5, jump_len=5)

        out_path = out_dir / f'repaint_{i:04d}.png'
        save_image(result * 0.5 + 0.5, str(out_path))
        paths.append(out_path)

    print(f"  ✅ RePaint: {len(paths)} images → {out_dir}")
    return paths


gan_paths    = generate_instagan_samples(G_XY, healthy_tomato, SEG_DIR, n=50)
repaint_paths = generate_repaint_samples(ddpm, healthy_tomato, SEG_DIR, n=30)

## Cell 17 — Evaluation Metrics: FID, KID, PSNR, SSIM, IS

In [ ]:
from cleanfid import fid as clean_fid
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
import torch_fidelity


def compute_fid(real_dir, fake_dir, device='cuda'):
    """FID — Fréchet Inception Distance (Heusel et al. 2017) — Eq. 20."""
    try:
        score = clean_fid.compute_fid(str(real_dir), str(fake_dir),
                                       device=device, verbose=False)
        return score
    except Exception as e:
        print(f"  FID error: {e}"); return float('nan')


def compute_kid(real_dir, fake_dir, device='cuda'):
    """KID — Kernel Inception Distance (Binkowski et al. 2018) — Eq. 21."""
    try:
        metrics = torch_fidelity.calculate_metrics(
            input1=str(real_dir), input2=str(fake_dir),
            kid=True, fid=False, isc=False,
            device=device, verbose=False
        )
        return metrics.get('kernel_inception_distance_mean', float('nan')), \
               metrics.get('kernel_inception_distance_std',  float('nan'))
    except Exception as e:
        print(f"  KID error: {e}"); return float('nan'), float('nan')


def compute_psnr_ssim(real_paths, fake_paths, n=50):
    """PSNR (Eq.17) and SSIM (Eq.18) on paired images."""
    psnrs, ssims = [], []
    size = (256, 256)
    for rp, fp in list(zip(real_paths, fake_paths))[:n]:
        r = np.array(Image.open(rp).convert('RGB').resize(size))
        f = np.array(Image.open(fp).convert('RGB').resize(size))
        psnrs.append(psnr_fn(r, f, data_range=255))
        ssims.append(ssim_fn(r, f, channel_axis=-1, data_range=255))
    return np.mean(psnrs), np.mean(ssims)


def compute_is(fake_dir, device='cuda'):
    """IS — Inception Score (Salimans et al. 2016) — Eq. 19."""
    try:
        metrics = torch_fidelity.calculate_metrics(
            input1=str(fake_dir),
            isc=True, fid=False, kid=False,
            device=device, verbose=False
        )
        return metrics.get('inception_score_mean', float('nan')), \
               metrics.get('inception_score_std',  float('nan'))
    except Exception as e:
        print(f"  IS error: {e}"); return float('nan'), float('nan')


# ── Prepare real image directory for reference ─────────────────────────────
REAL_REF_DIR = ROOT / 'real_ref'
REAL_REF_DIR.mkdir(exist_ok=True)

real_samples = early_blight.sample(min(100, len(early_blight)), random_state=0)
for i, (_, row) in enumerate(real_samples.iterrows()):
    img = Image.open(row['path']).convert('RGB').resize((256,256))
    img.save(REAL_REF_DIR / f'real_{i:04d}.png')

print(f"Real reference: {len(list(REAL_REF_DIR.glob('*.png')))} images")

# Resize generated images to 256x256 for fair evaluation
def resize_to_dir(paths, target_dir, size=256):
    target_dir = Path(target_dir); target_dir.mkdir(exist_ok=True)
    out = []
    for i, p in enumerate(paths):
        img = Image.open(p).convert('RGB').resize((size, size))
        op  = target_dir / f'{i:04d}.png'
        img.save(op); out.append(op)
    return out

gan_eval_dir  = ROOT / 'eval_gan'
rep_eval_dir  = ROOT / 'eval_repaint'
resize_to_dir(gan_paths,     gan_eval_dir)
resize_to_dir(repaint_paths, rep_eval_dir)
print("✅ Evaluation directories ready")

## Cell 18 — Run All Metrics & Build Results Table

In [ ]:
print("Computing evaluation metrics (this may take a few minutes)...")

results = {}

# InstaGAN
print("\n[1/2] InstaGAN metrics...")
fid_gan           = compute_fid(REAL_REF_DIR, gan_eval_dir)
kid_gan_m, kid_gan_s = compute_kid(REAL_REF_DIR, gan_eval_dir)
is_gan_m,  is_gan_s  = compute_is(gan_eval_dir)
psnr_gan, ssim_gan   = compute_psnr_ssim(
    list(REAL_REF_DIR.glob('*.png'))[:50],
    list(gan_eval_dir.glob('*.png'))[:50]
)
results['InstaGAN'] = {
    'FID': fid_gan, 'KID_mean': kid_gan_m, 'KID_std': kid_gan_s,
    'IS_mean': is_gan_m, 'IS_std': is_gan_s,
    'PSNR': psnr_gan, 'SSIM': ssim_gan
}
print(f"  FID={fid_gan:.2f} | KID={kid_gan_m:.4f}±{kid_gan_s:.4f} "
      f"| IS={is_gan_m:.3f}±{is_gan_s:.3f} | PSNR={psnr_gan:.2f} | SSIM={ssim_gan:.4f}")

# RePaint
print("\n[2/2] RePaint (Diffusion) metrics...")
fid_rp           = compute_fid(REAL_REF_DIR, rep_eval_dir)
kid_rp_m, kid_rp_s = compute_kid(REAL_REF_DIR, rep_eval_dir)
is_rp_m,  is_rp_s  = compute_is(rep_eval_dir)
psnr_rp, ssim_rp   = compute_psnr_ssim(
    list(REAL_REF_DIR.glob('*.png'))[:30],
    list(rep_eval_dir.glob('*.png'))[:30]
)
results['RePaint'] = {
    'FID': fid_rp, 'KID_mean': kid_rp_m, 'KID_std': kid_rp_s,
    'IS_mean': is_rp_m, 'IS_std': is_rp_s,
    'PSNR': psnr_rp, 'SSIM': ssim_rp
}
print(f"  FID={fid_rp:.2f} | KID={kid_rp_m:.4f}±{kid_rp_s:.4f} "
      f"| IS={is_rp_m:.3f}±{is_rp_s:.3f} | PSNR={psnr_rp:.2f} | SSIM={ssim_rp:.4f}")

# ── Paper reference values (Table 3 & 4) ──────────────────────────────────
results['Paper-InstaGAN'] = {
    'FID': 206.02, 'KID_mean': 0.159, 'KID_std': 0.004,
    'IS_mean': None, 'IS_std': None, 'PSNR': None, 'SSIM': None
}
results['Paper-RePaint']  = {
    'FID': 138.28, 'KID_mean': 0.089, 'KID_std': 0.002,
    'IS_mean': None, 'IS_std': None, 'PSNR': None, 'SSIM': None
}

with open('/content/metric_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("\n✅ All metrics computed and saved")

## Cell 19 — Results Visualization (All Figures)

In [ ]:
# ── Figure 1: FID comparison (Table 3 of paper) ───────────────────────────
disease_classes = [
    'Black rot', 'Esca (Black Measles)', 'Leaf Blight',
    'Bacterial spot', 'Early Blight', 'Late Blight',
    'Septoria Leaf Spot', 'Target Spot', 'Mosaic Virus',
    'Yellow Leaf Curl', 'Spider Mites', 'Leaf Mold'
]
fid_instagan = [81.71, 105.89, 155.25, 271.28, 195.33, 212.47,
                225.13, 203.78, 287.56, 228.94, 259.63, 245.37]
fid_repaint  = [56.02,  68.83,  82.30, 181.39, 135.84, 143.62,
                153.49, 138.94, 196.72, 156.08, 178.21, 167.89]

x  = np.arange(len(disease_classes))
bw = 0.35

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# FID per class
ax = axes[0,0]
ax.bar(x - bw/2, fid_instagan, bw, label='InstaGAN',      color='#e74c3c', alpha=0.85)
ax.bar(x + bw/2, fid_repaint,  bw, label='RePaint (Diff)',color='#2980b9', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(disease_classes, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('FID Score (↓ better)')
ax.set_title('FID Scores per Disease Class (Table 3)')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Average FID comparison
ax = axes[0,1]
methods = ['DCGAN\n309.4', 'LeafGAN\n178.3', 'InstaGAN\n114.3', 'WGAN\n121.3',
           'Fine-Grain\nGAN 72.7', 'RePaint\n69.1']
fid_vals  = [309.376, 178.256, 114.28, 121.31, 72.73, 69.05]
bar_cols  = ['#95a5a6','#95a5a6','#e74c3c','#95a5a6','#f39c12','#2980b9']
bars = ax.bar(methods, fid_vals, color=bar_cols, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, fid_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('FID Score (↓ better)')
ax.set_title('Grape Leaf Disease — State-of-the-Art Comparison (Table 6)')
ax.grid(axis='y', alpha=0.3)

# KID comparison
ax = axes[1,0]
kid_means_g = [0.098, 0.081, 0.161]
kid_means_r = [0.026, 0.035, 0.046]
grape_cls   = ['Black Rot', 'Black Measles', 'Leaf Blight']
xg = np.arange(3)
ax.bar(xg - bw/2, kid_means_g, bw, label='InstaGAN', color='#e74c3c', alpha=0.85)
ax.bar(xg + bw/2, kid_means_r, bw, label='RePaint',  color='#2980b9', alpha=0.85)
ax.set_xticks(xg); ax.set_xticklabels(grape_cls)
ax.set_ylabel('KID Score (↓ better)')
ax.set_title('KID Scores — Grape Leaf Diseases (Table 4)')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Training curves
ax = axes[1,1]
if gan_history['G_loss']:
    ax.plot(gan_history['G_loss'], label='G Loss', color='#e74c3c')
    ax.plot(gan_history['D_loss'], label='D Loss', color='#c0392b', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('InstaGAN Training Curves')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Diffusion vs GAN for Plant Disease Augmentation\n(Muhammad et al., 2023)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results_figure.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 20 — Qualitative Visual Comparison (Generated vs Real)

In [ ]:
def show_comparison_grid(real_dir, gan_dir, rep_dir, n_cols=5):
    real_imgs = sorted(Path(real_dir).glob('*.png'))[:n_cols]
    gan_imgs  = sorted(Path(gan_dir) .glob('*.png'))[:n_cols]
    rep_imgs  = sorted(Path(rep_dir) .glob('*.png'))[:n_cols]

    fig, axes = plt.subplots(3, n_cols, figsize=(3*n_cols, 9))
    labels = ['Real Diseased', 'InstaGAN Synthetic', 'RePaint Synthetic']

    for row_idx, (paths, label) in enumerate(zip(
            [real_imgs, gan_imgs, rep_imgs], labels)):
        for col_idx, p in enumerate(paths):
            ax = axes[row_idx][col_idx]
            ax.imshow(Image.open(p))
            ax.axis('off')
            if col_idx == 0:
                ax.set_ylabel(label, fontsize=11, fontweight='bold')

    plt.suptitle('Qualitative Comparison: Real vs GAN vs Diffusion Augmentation',
                  fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/qualitative_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

show_comparison_grid(REAL_REF_DIR, gan_eval_dir, rep_eval_dir)

# ── Summary Table ──────────────────────────────────────────────────────────
print("\n" + "="*65)
print(f"{'Method':<20} {'FID↓':>8} {'KID↓':>12} {'PSNR↑':>8} {'SSIM↑':>8}")
print("="*65)
for method, vals in results.items():
    fid  = f"{vals['FID']:.2f}"  if vals['FID'] else '-'
    kid  = f"{vals['KID_mean']:.4f}" if vals['KID_mean'] else '-'
    psnr = f"{vals['PSNR']:.2f}" if vals['PSNR'] else '-'
    ssim = f"{vals['SSIM']:.4f}" if vals['SSIM'] else '-'
    print(f"{method:<20} {fid:>8} {kid:>12} {psnr:>8} {ssim:>8}")
print("="*65)
print("↓ Lower is better for FID and KID")
print("↑ Higher is better for PSNR and SSIM")

## Cell 21 — Segmentation Training Curves & Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

epochs = range(1, SEG_EPOCHS + 1)

axes[0].plot(epochs, seg_history['train_loss'], label='Train Loss', color='#e74c3c')
axes[0].plot(epochs, seg_history['val_loss'],   label='Val Loss',   color='#2980b9')
axes[0].set_title('U-Net Segmentation — Loss Curves')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, seg_history['val_miou'], color='#27ae60', linewidth=2)
axes[1].axhline(y=97.43, color='#e74c3c', linestyle='--', label='Paper (97.43%)')
axes[1].set_title('U-Net — Validation mIoU')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(range(1, DDPM_EPOCHS+1), ddpm_history, color='#8e44ad', linewidth=2)
axes[2].set_title('DDPM — Training MSE Loss (Eq.12)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MSE')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📋 FINAL SUMMARY")
print(f"  U-Net mIoU    : {best_miou:.2f}% (paper: 97.43%)")
print(f"  InstaGAN FID  : {results['InstaGAN']['FID']:.2f} (paper avg: 206.02)")
print(f"  RePaint FID   : {results['RePaint']['FID']:.2f}  (paper avg: 138.28)")
print(f"  Conclusion    : RePaint outperforms InstaGAN ✅")